# Connect

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/KLTN

/content/drive/MyDrive/KLTN


In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.5 MB/s eta 0:00:00


In [ ]:
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Input, concatenate, Reshape, Conv1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf



# Multimodal TransferLearning

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = x.relu()
        x = global_mean_pool(x, batch)  # Pooling layer
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        x = torch.sigmoid(x)  # Sigmoid activation for multi-label classification
        return x


In [ ]:
import pandas as pd
import numpy as np
from numpy import savetxt, loadtxt

def load_dataset(label_count):
    # Tải dữ liệu từ file CSV
    data = pd.read_csv("/content/drive/MyDrive/KLTN/DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0', axis=1)

    # Các cột nhãn
    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Lấy số lượng hàng dữ liệu là 39645
    data = data.iloc[:39645]

    # Chia tập dữ liệu thành tập huấn luyện và tập kiểm tra
    total_rows = len(data)
    split_point = int(0.8 * total_rows)
    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    features_source_train =  np.loadtxt("/content/drive/MyDrive/KLTN/DATASET/split_train.csv", delimiter=',')
    features_source_test = np.loadtxt("/content/drive/MyDrive/KLTN/DATASET/split_test.csv", delimiter=',')
    X_train_source = np.array(features_source_train)
    X_test_source = np.array(features_source_test)
    print("Load Features codeBERT success!!")
    # Lấy các đặc trưng từ tập dữ liệu
    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # Lấy nhãn từ tập dữ liệu
    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels_dataset = data[selected_labels].values
    # Load data
    input_file = '/content/drive/MyDrive/KLTN/DATASET/gnn_input_binary_labels.pkl'

    with open(input_file, 'rb') as f:
        data = pickle.load(f)

    loaded_feature_matrices = data['features']
    adj_matrices = data['adj_matrices']
    # loaded_all_labels = data['labels']
    # Define the target dimension
    target_dim = 6000

    def pad_or_truncate_features(features, target_dim):
        if features.shape[1] > target_dim:
            return features[:, :target_dim]
        elif features.shape[1] < target_dim:
            padded_features = np.zeros((features.shape[0], target_dim))
            padded_features[:, :features.shape[1]] = features
            return padded_features
        else:
            return features

    # Function to create PyTorch Geometric Data object from feature matrices and labels
    def create_pyg_data(features, adj_matrix, labels, target_dim):
        num_nodes = features.shape[0]
        edge_index = torch.tensor(np.array(adj_matrix), dtype=torch.long).nonzero(as_tuple=False).t().contiguous()
        x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
        y = torch.tensor(labels, dtype=torch.float)  # Ensure labels are float for BCEWithLogitsLoss
        data = Data(x=x, edge_index=edge_index, y=y)
        return data

    data_list = [create_pyg_data(features, adj_matrix, labels, target_dim) for features, adj_matrix, labels in zip(loaded_feature_matrices, adj_matrices, labels_dataset)]

    # Create DataLoader
    train_loader = DataLoader(data_list[:int(len(data_list)*0.8)], batch_size=32, shuffle=True)  # Adjust batch size as needed
    test_loader = DataLoader(data_list[int(len(data_list)*0.8):], batch_size=32, shuffle=False)  # Adjust batch size as needed

    print(f"Load Data for {label_count}-Label MultiLabel success!!")
    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, train_loader, test_loader,  y_train, y_test, labels_dataset



In [ ]:
X_train_opcode_4, X_test_opcode_4, X_train_source_4, X_test_source_4, train_loader_4, test_loader_4,  y_train_4, y_test_4, labels_4 = load_dataset(4)

Load Features codeBERT success!!
Load Data for 4-Label MultiLabel success!!


In [ ]:
X_train_opcode_8, X_test_opcode_8, X_train_source_8, X_test_source_8, train_loader_8, test_loader_8,  y_train_8, y_test_8, labels_8 = load_dataset(8)

Load Features codeBERT success!!
Load Data for 8-Label MultiLabel success!!


In [ ]:
def train_and_save_model(in_channels, hidden_channels, out_channels, train_loader, save_path):
    model = GCN(in_channels, hidden_channels, out_channels)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.BCELoss()

    model.train()
    for epoch in range(30):  # Adjust the number of epochs as needed
        for data in train_loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y.view(out.size()))  # Ensure the target size matches the output size
            loss.backward()
            optimizer.step()
            print(f"Epoch {epoch+1}/{30}, Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), save_path)

# Example: Train and save models for 4 and 8 labels
train_and_save_model(in_channels=6000, hidden_channels=64, out_channels=4, train_loader=train_loader_4, save_path='/content/drive/MyDrive/KLTN/Unimodal/GCN_30E_4_labels.pth')
train_and_save_model(in_channels=6000, hidden_channels=64, out_channels=8, train_loader=train_loader_8, save_path='/content/drive/MyDrive/KLTN/Unimodal/GCN_30E_8_labels.pth')

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
Epoch 25/30, Loss: 0.5447
Epoch 25/30, Loss: 0.6159
Epoch 25/30, Loss: 0.4801
Epoch 25/30, Loss: 0.4922
Epoch 25/30, Loss: 0.5180
Epoch 25/30, Loss: 0.4903
Epoch 25/30, Loss: 0.4928
Epoch 25/30, Loss: 0.5264
Epoch 25/30, Loss: 0.4715
Epoch 25/30, Loss: 0.5010
Epoch 25/30, Loss: 0.4897
Epoch 25/30, Loss: 0.4838
Epoch 25/30, Loss: 0.5247
Epoch 25/30, Loss: 0.4305
Epoch 25/30, Loss: 0.4542
Epoch 25/30, Loss: 0.4921
Epoch 25/30, Loss: 0.4867
Epoch 25/30, Loss: 0.4763
Epoch 25/30, Loss: 0.4594
Epoch 25/30, Loss: 0.5311
Epoch 25/30, Loss: 0.5022
Epoch 25/30, Loss: 0.4943
Epoch 25/30, Loss: 0.5480
Epoch 25/30, Loss: 0.5217
Epoch 25/30, Loss: 0.4949
Epoch 25/30, Loss: 0.4671
Epoch 25/30, Loss: 0.4729
Epoch 25/30, Loss: 0.4649
Epoch 25/30, Loss: 0.4832
Epoch 25/30, Loss: 0.5473
Epoch 25/30, Loss: 0.4672
Epoch 25/30, Loss: 0.4860
Epoch 25/30, Loss: 0.5502
Epoch 25/30, Loss: 0.5265
Epoch 25/30, Loss: 0.5074
Epoch 25/30, Loss: 0.5254
Epoch 2

In [ ]:
def load_model(model_path, in_channels, hidden_channels, out_channels):
    model = GCN(in_channels=in_channels, hidden_channels=hidden_channels, out_channels=out_channels)
    model.load_state_dict(torch.load(model_path))
    model.eval()  # Set the model to evaluation mode
    return model

def get_gnn_outputs(loader, model):
    outputs = []
    with torch.no_grad():
        for data in loader:
            out = model(data)
            outputs.append(out.numpy())
    return np.concatenate(outputs)

def save_gnn_outputs(outputs, file_path):
    np.save(file_path, outputs)

def load_gnn_outputs(file_path):
    return np.load(file_path)

# Example usage
model_paths = {
    4: '/content/drive/MyDrive/KLTN/Unimodal/GCN_30E_4_labels.pth',
    8: '/content/drive/MyDrive/KLTN/Unimodal/GCN_30E_8_labels.pth'
}
in_channels = 6000
hidden_channels = 64

def run_gnn_model(num_labels, train_loader, test_loader):
    # Load the model with the specified number of labels (out_channels)
    model = load_model(model_paths[num_labels], in_channels, hidden_channels, num_labels)

    # Get GNN outputs for training and testing data
    train_outputs = get_gnn_outputs(train_loader, model)
    test_outputs = get_gnn_outputs(test_loader, model)

    return train_outputs, test_outputs

In [ ]:
train_outputs_8, test_outputs_8 = run_gnn_model(num_labels=8, train_loader=train_loader_8, test_loader=test_loader_8)
train_outputs_4, test_outputs_4 = run_gnn_model(num_labels=4, train_loader=train_loader_4, test_loader=test_loader_4)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Conv1D, Flatten, Reshape, concatenate
from tensorflow.keras.models import load_model

def create_model_with_labels(initial_label_count, total_label_count):
    # Load Bert, Bi-LSTM Model
    bert_model = load_model("/content/drive/MyDrive/KLTN/Unimodal/CodeBERT_30E.h5", compile=False)
    lstm_model = load_model("/content/drive/MyDrive/KLTN/Unimodal/BiLSTM_Full_2gram_Softmax_30E.h5", compile=False)
    # Set the models to non-trainable
    bert_model.trainable = False
    lstm_model.trainable = False

    # Define GNN output
    gnn_output = tf.keras.layers.Input(shape=(initial_label_count,), name="gnn_output")  # Output size of GNN

    # Combine the outputs of both models
    bert_output = bert_model.output
    lstm_output = lstm_model.output
    concatenated = concatenate([bert_output, lstm_output, gnn_output], axis=-1)

    # Feature extractor part
    concatenated_reshaped = Reshape((-1, 1))(concatenated)  # Adjust to match Conv1D input
    conv_out = Conv1D(24, 3, activation='relu')(concatenated_reshaped)
    flatten_out = Flatten()(conv_out)
    dense_features = Dense(128, activation='relu', name='dense_multilabel')(flatten_out)
    dense_features = Dropout(0.3, name='drop_multilabel')(dense_features)

    # Output for the initial_label_count task
    output_initial_labels = Dense(initial_label_count, activation='sigmoid', name='output_initial_labels')(dense_features)

    # Create initial model
    model_initial = tf.keras.models.Model(inputs=[bert_model.input, lstm_model.input, gnn_output], outputs=output_initial_labels)

    # Compile the model
    METRICS = [
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]

    model_initial.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

    return model_initial


In [ ]:
# Check if gnn_output is available
initial_label_count = 4
model_initial = create_model_with_labels(initial_label_count, total_label_count=8)


In [ ]:
history = model_initial.fit([ X_train_source_4, X_train_opcode_4, train_outputs_4], y_train_4, epochs=10, batch_size=32)

Epoch 1/10
992/992 [==============================] - 122s 117ms/step - loss: 0.2702 - accuracy: 0.8939 - precision: 0.8808 - recall: 0.8868
Epoch 2/10
992/992 [==============================] - 117s 118ms/step - loss: 0.2442 - accuracy: 0.9036 - precision: 0.8946 - recall: 0.8934
Epoch 3/10
992/992 [==============================] - 110s 111ms/step - loss: 0.2415 - accuracy: 0.9041 - precision: 0.8973 - recall: 0.8914
Epoch 4/10
992/992 [==============================] - 110s 111ms/step - loss: 0.2405 - accuracy: 0.9042 - precision: 0.8974 - recall: 0.8914
Epoch 5/10
992/992 [==============================] - 111s 112ms/step - loss: 0.2385 - accuracy: 0.9050 - precision: 0.8987 - recall: 0.8915
Epoch 6/10
992/992 [==============================] - 108s 109ms/step - loss: 0.2369 - accuracy: 0.9052 - precision: 0.8985 - recall: 0.8925
Epoch 7/10
992/992 [==============================] - 108s 109ms/step - loss: 0.2387 - accuracy: 0.9044 - precision: 0.8982 - recall: 0.8908
Epoch 8/10
99

In [ ]:
model_initial.save('Weighted_model/Multimodal_3Component_4Label.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
history = model_initial.fit([ X_train_source_4, X_train_opcode_4, train_outputs_4], y_train_4, epochs=20, batch_size=32)

Epoch 1/20
992/992 [==============================] - 110s 111ms/step - loss: 0.2361 - accuracy: 0.9049 - precision: 0.9000 - recall: 0.8899
Epoch 2/20
992/992 [==============================] - 109s 110ms/step - loss: 0.2360 - accuracy: 0.9049 - precision: 0.9002 - recall: 0.8895
Epoch 3/20
992/992 [==============================] - 108s 109ms/step - loss: 0.2359 - accuracy: 0.9049 - precision: 0.9001 - recall: 0.8898
Epoch 4/20
992/992 [==============================] - 108s 108ms/step - loss: 0.2349 - accuracy: 0.9049 - precision: 0.8999 - recall: 0.8898
Epoch 5/20
992/992 [==============================] - 108s 109ms/step - loss: 0.2351 - accuracy: 0.9052 - precision: 0.8988 - recall: 0.8919
Epoch 6/20
992/992 [==============================] - 111s 112ms/step - loss: 0.2344 - accuracy: 0.9041 - precision: 0.8979 - recall: 0.8905
Epoch 7/20
992/992 [==============================] - 111s 112ms/step - loss: 0.2360 - accuracy: 0.9046 - precision: 0.8989 - recall: 0.8904
Epoch 8/20
99

In [ ]:
model_initial.save('Weighted_model/Multimodal_3Component_30E_4Label.h5')

In [ ]:
def extend_model_with_labels(model_initial, initial_label_count, total_label_count):
    # Get inputs and the concatenated output
    bert_input = model_initial.input[0]
    lstm_input = model_initial.input[1]
    gnn_input = model_initial.input[2]
    concatenated = model_initial.get_layer('concatenate_1').output  # Get the output of the concatenation layer

    # Feature extractor part
    concatenated_reshaped = Reshape((-1, 1), name='reshape_2')(concatenated)
    conv_out = Conv1D(24, 3, activation='relu', name='conv1d_2')(concatenated_reshaped)
    flatten_out = Flatten(name='flatten_2')(conv_out)
    dense_features = Dense(128, activation='relu', name='dense_multilabel_new')(flatten_out)
    dense_features = Dropout(0.3, name='drop_multilabel_new')(dense_features)

    # Output for the total_label_count task
    output_total_labels = Dense(total_label_count, activation='sigmoid', name='output_total_labels')(dense_features)

    # Create new model
    model_new = tf.keras.models.Model(inputs=[bert_input, lstm_input, gnn_input], outputs=output_total_labels)

    # Copy weights from the initial model to the new model
    model_new.get_layer('dense_multilabel_new').set_weights(model_initial.get_layer('dense_multilabel').get_weights())
    model_new.get_layer('drop_multilabel_new').set_weights(model_initial.get_layer('drop_multilabel').get_weights())

    # Set the output weights
    initial_weights = model_initial.get_layer('output_initial_labels').get_weights()
    new_weights = model_new.get_layer('output_total_labels').get_weights()

    new_weights[0][:, :initial_label_count] = initial_weights[0]  # Copy the weights for the initial labels
    new_weights[1][:initial_label_count] = initial_weights[1]  # Copy the biases for the initial labels

    model_new.get_layer('output_total_labels').set_weights(new_weights)

    # Compile the new model
    METRICS = [
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]

    model_new.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

    return model_new

In [ ]:
# Example usage to extend to 8 labels
model_new = extend_model_with_labels(model_initial, initial_label_count=4, total_label_count=8)
model_new.summary()

ValueError: No such layer: concatenate_1. Existing layers are: ['input_1', 'sourcecode_features', 'embedding', 'codebert_dense_layer_1', 'bidirectional', 'codebert_bn_1', 'dropout', 'codebert_dropout_layer_1', 'batch_normalization1', 'codebert_dense_layer_2', 'drop_out', 'codebert_bn_2', 'dense', 'codebert_dropout_layer_2', 'batch_normalization', 'codebert_dense_layer_out', 'dropout_1', 'codebert_bn', 'dense_1', 'codebert_output', 'dense_2', 'gnn_output', 'concatenate_2', 'reshape_2', 'conv1d_2', 'flatten_2', 'dense_multilabel', 'drop_multilabel', 'output_initial_labels'].

Transfer Leaning

In [ ]:
# Load GNN
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np

# Load data
input_file = '/content/drive/MyDrive/KLTN/DATASET/gnn_input_binary_labels.pkl'

with open(input_file, 'rb') as f:
    data = pickle.load(f)

loaded_feature_matrices = data['features']
adj_matrices = data['adj_matrices']
# loaded_all_labels = data['labels']
# Define the target dimension
target_dim = 6000

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim))
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features

# Function to create PyTorch Geometric Data object from feature matrices and labels
def create_pyg_data(features, adj_matrix, labels, target_dim):
    num_nodes = features.shape[0]
    edge_index = torch.tensor(np.array(adj_matrix), dtype=torch.long).nonzero(as_tuple=False).t().contiguous()
    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.float)  # Ensure labels are float for BCEWithLogitsLoss
    data = Data(x=x, edge_index=edge_index, y=y)
    return data

data_list = [create_pyg_data(features, adj_matrix, labels, target_dim) for features, adj_matrix, labels in zip(loaded_feature_matrices, adj_matrices, labels_dataset)]

# Create DataLoader
train_loader = DataLoader(data_list[:int(len(data_list)*0.8)], batch_size=32, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(data_list[int(len(data_list)*0.8):], batch_size=32, shuffle=False)  # Adjust batch size as needed


In [ ]:
import torch
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
import pickle

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = x.relu()
        x = global_mean_pool(x, batch)  # Pooling layer
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        x = torch.sigmoid(x)  # Sigmoid activation for multi-label classification
        return x

# Load mô hình GNN
model_path = '/content/drive/MyDrive/KLTN/Unimodal/GCN_60E.pth'

# Adjust the in_channels, hidden_channels, and out_channels parameters based on the training settings
gnn_model = GCN(in_channels=6000, hidden_channels=64, out_channels=8)
gnn_model.load_state_dict(torch.load(model_path))
gnn_model.eval()  # Set the model to evaluation mode

# Function to get GNN outputs for a given DataLoader
def get_gnn_outputs(loader):
    outputs = []
    with torch.no_grad():
        for data in loader:
            out = gnn_model(data)
            outputs.append(out.numpy())
    return np.concatenate(outputs)

# Get GNN outputs for training and testing data
gnn_train_outputs = get_gnn_outputs(train_loader)
gnn_test_outputs = get_gnn_outputs(test_loader)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Conv1D, Flatten, Reshape, concatenate
from tensorflow.keras.models import load_model

def create_model_with_labels(initial_label_count, total_label_count):
    # Load Bert, Bi-LSTM Model
    bert_model = load_model("/content/drive/MyDrive/KLTN/Unimodal/CodeBERT_30E.h5", compile=False)
    lstm_model = load_model("/content/drive/MyDrive/KLTN/Unimodal/BiLSTM_Full_2gram_Softmax_30E.h5", compile=False)
    # Set the models to non-trainable
    bert_model.trainable = False
    lstm_model.trainable = False

    # Define GNN output
    gnn_output = tf.keras.layers.Input(shape=(8,), name="gnn_output")  # Output size of GNN

    # Combine the outputs of both models
    bert_output = bert_model.output
    lstm_output = lstm_model.output
    concatenated = concatenate([bert_output, lstm_output, gnn_output], axis=-1)

    # Feature extractor part
    concatenated_reshaped = Reshape((-1, 1))(concatenated)  # Adjust to match Conv1D input
    conv_out = Conv1D(24, 3, activation='relu')(concatenated_reshaped)
    flatten_out = Flatten()(conv_out)
    dense_features = Dense(128, activation='relu', name='dense_multilabel')(flatten_out)
    dense_features = Dropout(0.3, name='drop_multilabel')(dense_features)

    # Output for the initial_label_count task
    output_initial_labels = Dense(initial_label_count, activation='sigmoid', name='output_initial_labels')(dense_features)

    # Create initial model
    model_initial = tf.keras.models.Model(inputs=[bert_model.input, lstm_model.input, gnn_output], outputs=output_initial_labels)

    # Compile the model
    METRICS = [
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]

    model_initial.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

    return model_initial


In [ ]:
# Check if gnn_output is available
initial_label_count = 4
try:
    # Create model with labels
    model_initial = create_model_with_labels(initial_label_count, total_label_count=8)
    model_initial.summary()
except ValueError as e:
    if "cannot obtain value for tensor" in str(e):
        print("Error: GNN output not accessible in the model.")
        # Further investigate the issue with the model or input data
    else:
        raise e

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 280)]                0         []                            
                                                                                                  
 sourcecode_features (Input  [(None, 768)]                0         []                            
 Layer)                                                                                           
                                                                                                  
 embedding (Embedding)       (None, 280, 286)             9724000   ['input_1[0][0]']             
                                                                                                  
 codebert_dense_layer_1 (De  (None, 512)                  393728    ['sourcecode_features[0]

In [ ]:
print("X_train_source_4 shape:", X_train_source_4.shape)
print("X_train_opcode_4 shape:", X_train_opcode_4.shape)
print("gnn_train_outputs shape:", gnn_train_outputs.shape)
print("y_train_4 shape:", y_train_4.shape)
gnn_train_outputs = np.array(gnn_train_outputs)
gnn_test_outputs = np.array(gnn_test_outputs)
tf.config.run_functions_eagerly(True)



# Call model_initial.fit()
history = model_initial.fit([ X_train_source_4, X_train_opcode_4, gnn_train_outputs], y_train_4, epochs=10, batch_size=32)

X_train_source_4 shape: (31716, 768)
X_train_opcode_4 shape: (31716, 280)
gnn_train_outputs shape: (31716, 8)
y_train_4 shape: (31716, 4)
Epoch 1/10


/usr/local/lib/python3.10/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


 17/992 [..............................] - ETA: 51:13 - loss: 0.5497 - accuracy: 0.7431 - precision: 0.7279 - recall: 0.6933

KeyboardInterrupt: 

# Test 1

In [ ]:
model_multimodal = load_model('Multimodal_3component.h5', compile=False)

In [ ]:
def load_dataset(label_count):
    # Tải dữ liệu từ file CSV
    data = pd.read_csv("/content/drive/MyDrive/KLTN/DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0', axis=1)

    # Các cột nhãn
    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Lấy số lượng hàng dữ liệu là 39645
    data = data.iloc[:39645]

    # Chia tập dữ liệu thành tập huấn luyện và tập kiểm tra
    total_rows = len(data)
    split_point = int(0.8 * total_rows)
    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    features_source_train =  np.loadtxt("/content/drive/MyDrive/KLTN/DATASET/split_train.csv", delimiter=',')
    features_source_test = np.loadtxt("/content/drive/MyDrive/KLTN/DATASET/split_test.csv", delimiter=',')
    X_train_source = np.array(features_source_train)
    X_test_source = np.array(features_source_test)
    print("Load Features codeBERT success!!")
    # Lấy các đặc trưng từ tập dữ liệu
    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # Lấy nhãn từ tập dữ liệu
    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")

    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels

In [ ]:
X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels_dataset = load_dataset(8)

Load Features codeBERT success!!
Load Data for 8-Label MultiLabel success!!


In [ ]:
# Load GNN
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np

# Load data
input_file = '/content/drive/MyDrive/KLTN/DATASET/gnn_input_binary_labels.pkl'

with open(input_file, 'rb') as f:
    data = pickle.load(f)

loaded_feature_matrices = data['features']
adj_matrices = data['adj_matrices']
# loaded_all_labels = data['labels']
# Define the target dimension
target_dim = 6000

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim))
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features

# Function to create PyTorch Geometric Data object from feature matrices and labels
def create_pyg_data(features, adj_matrix, labels, target_dim):
    num_nodes = features.shape[0]
    edge_index = torch.tensor(np.array(adj_matrix), dtype=torch.long).nonzero(as_tuple=False).t().contiguous()
    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.float)  # Ensure labels are float for BCEWithLogitsLoss
    data = Data(x=x, edge_index=edge_index, y=y)
    return data

data_list = [create_pyg_data(features, adj_matrix, labels, target_dim) for features, adj_matrix, labels in zip(loaded_feature_matrices, adj_matrices, labels_dataset)]

# Create DataLoader
train_loader = DataLoader(data_list[:int(len(data_list)*0.8)], batch_size=32, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(data_list[int(len(data_list)*0.8):], batch_size=32, shuffle=False)  # Adjust batch size as needed


In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = x.relu()
        x = global_mean_pool(x, batch)  # Pooling layer
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        x = torch.sigmoid(x)  # Sigmoid activation for multi-label classification
        return x

# Load mô hình GNN
model_path = '/content/drive/MyDrive/KLTN/Unimodal/GCN_60E.pth'

# Adjust the in_channels, hidden_channels, and out_channels parameters based on the training settings
gnn_model = GCN(in_channels=6000, hidden_channels=64, out_channels=8)
gnn_model.load_state_dict(torch.load(model_path))
gnn_model.eval()  # Set the model to evaluation mode

# Function to get GNN outputs for a given DataLoader
def get_gnn_outputs(loader):
    outputs = []
    with torch.no_grad():
        for data in loader:
            out = gnn_model(data)
            outputs.append(out.numpy())
    return np.concatenate(outputs)

# Get GNN outputs for training and testing data
gnn_train_outputs = get_gnn_outputs(train_loader)
gnn_test_outputs = get_gnn_outputs(test_loader)

In [ ]:
gnn_train_outputs

array([[8.4537256e-01, 1.6757995e-03, 9.9433374e-01, ..., 2.9288796e-03,
        1.5156448e-03, 3.2316335e-03],
       [9.9224186e-01, 4.0447838e-03, 9.9116546e-01, ..., 8.6100111e-03,
        3.8162810e-03, 1.0056260e-02],
       [6.6666335e-02, 1.0525585e-03, 9.9494451e-01, ..., 1.6485300e-03,
        9.2161272e-04, 1.5520577e-03],
       ...,
       [5.6446578e-02, 6.5566698e-04, 9.9658173e-01, ..., 1.1021149e-03,
        5.6599273e-04, 9.9302270e-04],
       [9.7779298e-01, 2.4450611e-04, 9.9888641e-01, ..., 4.8779760e-04,
        2.1303739e-04, 6.0508988e-04],
       [9.3866420e-01, 4.8429042e-04, 9.9794871e-01, ..., 8.8540575e-04,
        4.2603951e-04, 1.0569874e-03]], dtype=float32)

In [ ]:
def save_gnn_outputs_npy(outputs, file_path):
    np.save(file_path, outputs)
# Load functions
def load_gnn_outputs_npy(file_path):
    return np.load(file_path)


In [ ]:
save_gnn_outputs_npy(gnn_train_outputs, 'gnn_train_outputs_4.npy')
save_gnn_outputs_npy(gnn_test_outputs, 'gnn_test_outputs_4.npy')

In [ ]:
y_pred_raw = model_multimodal.predict([X_test_source, X_test_opcode, gnn_test_outputs])

248/248 [==============================] - 4s 8ms/step


In [ ]:
y_test[:1]

array([[1, 0, 1, 1, 0, 0, 0, 0]])

## Train model

# Load dữ liệu

In [ ]:
import pandas as pd
import numpy as np
from numpy import savetxt, loadtxt

def load_dataset(label_count):
    # Tải dữ liệu từ file CSV
    data = pd.read_csv("/content/drive/MyDrive/KLTN/DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0', axis=1)

    # Các cột nhãn
    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Lấy số lượng hàng dữ liệu là 39645
    data = data.iloc[:39645]

    # Chia tập dữ liệu thành tập huấn luyện và tập kiểm tra
    total_rows = len(data)
    split_point = int(0.8 * total_rows)
    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    features_source_train =  np.loadtxt("/content/drive/MyDrive/KLTN/DATASET/split_train.csv", delimiter=',')
    features_source_test = np.loadtxt("/content/drive/MyDrive/KLTN/DATASET/split_test.csv", delimiter=',')
    X_train_source = np.array(features_source_train)
    X_test_source = np.array(features_source_test)
    print("Load Features codeBERT success!!")
    # Lấy các đặc trưng từ tập dữ liệu
    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # Lấy nhãn từ tập dữ liệu
    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")
    # Load GNN

    # Load data
    input_file = '/content/drive/MyDrive/KLTN/DATASET/gnn_input_binary_labels.pkl'

    with open(input_file, 'rb') as f:
        data = pickle.load(f)

    loaded_feature_matrices = data['features']
    adj_matrices = data['adj_matrices']
    # loaded_all_labels = data['labels']
    # Define the target dimension
    target_dim = 6000

    def pad_or_truncate_features(features, target_dim):
        if features.shape[1] > target_dim:
            return features[:, :target_dim]
        elif features.shape[1] < target_dim:
            padded_features = np.zeros((features.shape[0], target_dim))
            padded_features[:, :features.shape[1]] = features
            return padded_features
        else:
            return features

    # Function to create PyTorch Geometric Data object from feature matrices and labels
    def create_pyg_data(features, adj_matrix, labels, target_dim):
        num_nodes = features.shape[0]
        edge_index = torch.tensor(np.array(adj_matrix), dtype=torch.long).nonzero(as_tuple=False).t().contiguous()
        x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
        y = torch.tensor(labels, dtype=torch.float)  # Ensure labels are float for BCEWithLogitsLoss
        data = Data(x=x, edge_index=edge_index, y=y)
        return data

    data_list = [create_pyg_data(features, adj_matrix, labels, target_dim) for features, adj_matrix, labels in zip(loaded_feature_matrices, adj_matrices, labels_dataset)]

    # Create DataLoader
    train_loader = DataLoader(data_list[:int(len(data_list)*0.8)], batch_size=32, shuffle=True)  # Adjust batch size as needed
    test_loader = DataLoader(data_list[int(len(data_list)*0.8):], batch_size=32, shuffle=False)  # Adjust batch size as needed


    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels



In [ ]:
# Ví dụ gọi hàm với số lượng nhãn là 8
X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels_dataset = load_dataset(8)

Load Features codeBERT success!!
Load Data for 8-Label MultiLabel success!!


In [ ]:
X_train_opcode.shape

(31716, 280)

In [ ]:
X_train_source.shape

(31716, 768)

In [ ]:
# Load GNN
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np

# Load data
input_file = '/content/drive/MyDrive/KLTN/DATASET/gnn_input_binary_labels.pkl'

with open(input_file, 'rb') as f:
    data = pickle.load(f)

loaded_feature_matrices = data['features']
adj_matrices = data['adj_matrices']
# loaded_all_labels = data['labels']
# Define the target dimension
target_dim = 6000

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim))
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features

# Function to create PyTorch Geometric Data object from feature matrices and labels
def create_pyg_data(features, adj_matrix, labels, target_dim):
    num_nodes = features.shape[0]
    edge_index = torch.tensor(np.array(adj_matrix), dtype=torch.long).nonzero(as_tuple=False).t().contiguous()
    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.float)  # Ensure labels are float for BCEWithLogitsLoss
    data = Data(x=x, edge_index=edge_index, y=y)
    return data

data_list = [create_pyg_data(features, adj_matrix, labels, target_dim) for features, adj_matrix, labels in zip(loaded_feature_matrices, adj_matrices, labels_dataset)]

# Create DataLoader
train_loader = DataLoader(data_list[:int(len(data_list)*0.8)], batch_size=32, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(data_list[int(len(data_list)*0.8):], batch_size=32, shuffle=False)  # Adjust batch size as needed


# Load model

In [ ]:
import torch
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
import pickle

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = x.relu()
        x = global_mean_pool(x, batch)  # Pooling layer
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        x = torch.sigmoid(x)  # Sigmoid activation for multi-label classification
        return x

# Load mô hình GNN
model_path = '/content/drive/MyDrive/KLTN/Unimodal/GCN_60E.pth'

# Adjust the in_channels, hidden_channels, and out_channels parameters based on the training settings
gnn_model = GCN(in_channels=6000, hidden_channels=64, out_channels=8)
gnn_model.load_state_dict(torch.load(model_path))
gnn_model.eval()  # Set the model to evaluation mode

# Function to get GNN outputs for a given DataLoader
def get_gnn_outputs(loader):
    outputs = []
    with torch.no_grad():
        for data in loader:
            out = gnn_model(data)
            outputs.append(out.numpy())
    return np.concatenate(outputs)

# Get GNN outputs for training and testing data
gnn_train_outputs = get_gnn_outputs(train_loader)
gnn_test_outputs = get_gnn_outputs(test_loader)

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Input, concatenate, Reshape, Conv1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
bert_model = load_model('Unimodal/CodeBERT_30E.h5', compile=False)
lstm_model = load_model('Unimodal/BiLSTM_Full_2gram_Softmax_30E.h5', compile=False)

# Tắt đi trainable của hai mô hình
bert_model.trainable = False
lstm_model.trainable = False
bert_output = bert_model.output
lstm_output = lstm_model.output
gnn_output = tf.keras.layers.Input(shape=(8,), name="gnn_output")  # Kích thước đầu ra của GNN

# Tắt trainable cho GNN
gnn_output._keras_history[0].trainable = False

OSError: No file or directory found at Unimodal/CodeBERT_30E.h5

In [ ]:
concatenated = concatenate([bert_output, lstm_output, gnn_output], axis=-1)
concatenated_reshaped = Reshape((24, 1))(concatenated)
conv_out = Conv1D(64, 3, activation='relu')(concatenated_reshaped)
flatten_out = Flatten()(conv_out)
dense_features = Dense(128, activation='relu', name='dense_multilabel')(flatten_out)
dense_features_2 = Dropout(0.3, name='drop_multilabel')(dense_features)
final_out = Dense(8, activation='sigmoid', name='output_multimodal')(dense_features_2)

In [ ]:
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]
# Tạo mô hình tổng cộng
model = tf.keras.models.Model(inputs=[bert_model.input, lstm_model.input, gnn_output], outputs=final_out)

# Compile mô hình
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

In [ ]:
model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 280)]                0         []                            
                                                                                                  
 sourcecode_features (Input  [(None, 768)]                0         []                            
 Layer)                                                                                           
                                                                                                  
 embedding (Embedding)       (None, 280, 286)             9724000   ['input_1[0][0]']             
                                                                                                  
 codebert_dense_layer_1 (De  (None, 512)                  393728    ['sourcecode_features[0]

In [ ]:

history = model.fit(
    [X_train_source, X_train_opcode, gnn_train_outputs],
    y_train,
    validation_data=([X_test_source, X_test_opcode, gnn_test_outputs], y_test),
    epochs=20,
    batch_size=32
)

Epoch 1/20
992/992 [==============================] - 119s 116ms/step - loss: 0.2693 - accuracy: 0.8931 - precision: 0.8534 - recall: 0.8203 - val_loss: 0.1723 - val_accuracy: 0.9383 - val_precision: 0.9017 - val_recall: 0.8928
Epoch 2/20
992/992 [==============================] - 116s 117ms/step - loss: 0.2470 - accuracy: 0.9006 - precision: 0.8597 - recall: 0.8388 - val_loss: 0.1640 - val_accuracy: 0.9386 - val_precision: 0.9113 - val_recall: 0.8822
Epoch 3/20
992/992 [==============================] - 114s 115ms/step - loss: 0.2459 - accuracy: 0.9011 - precision: 0.8625 - recall: 0.8368 - val_loss: 0.1634 - val_accuracy: 0.9391 - val_precision: 0.9126 - val_recall: 0.8826
Epoch 4/20
992/992 [==============================] - 114s 115ms/step - loss: 0.2432 - accuracy: 0.9017 - precision: 0.8664 - recall: 0.8337 - val_loss: 0.1610 - val_accuracy: 0.9387 - val_precision: 0.9148 - val_recall: 0.8786
Epoch 5/20
992/992 [==============================] - 115s 116ms/step - loss: 0.2417 - a

In [ ]:
model.save('Multimodal_3component.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
def apply_threshold(y_pred_prob, threshold=0.5):
    y_pred_prob = np.array(y_pred_prob)
    y_pred_thresh = np.zeros_like(y_pred_prob)
    y_pred_thresh[y_pred_prob >= threshold] = 1
    return y_pred_thresh

In [ ]:
y_pred_prob = model.predict([X_test_source, X_test_opcode, gnn_test_outputs])


248/248 [==============================] - 24s 94ms/step


In [ ]:
y_pred = apply_threshold(y_pred_prob, threshold=0.5)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
label_names = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

for i, label in enumerate(label_names):
    precision = precision_score(y_test[:, i], y_pred[:, i])
    recall = recall_score(y_test[:, i], y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    accuracy = accuracy_score(y_test[:, i], y_pred[:, i])
    # Tính confusion matrix cho nhãn i
    tn, fp, fn, tp = confusion_matrix(y_test[:, i], y_pred[:, i]).ravel()

    # Tính FPR và FNR
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    print(f"Metrics for {label}:")
    print(f"  Accuracy:  {accuracy:.2f}")
    print(f"  Precision: {precision:.2f}")
    print(f"  Recall:    {recall:.2f}")
    print(f"  F1-Score:  {f1:.2f}")
    print(f"  FPR:       {fpr:.2f}")
    print(f"  FNR:       {fnr:.2f}")
    print()

Metrics for Other:
  Accuracy:  0.88
  Precision: 0.89
  Recall:    0.90
  F1-Score:  0.90
  FPR:       0.15
  FNR:       0.10

Metrics for access_control:
  Accuracy:  0.96
  Precision: 0.94
  Recall:    0.54
  F1-Score:  0.68
  FPR:       0.00
  FNR:       0.46

Metrics for arithmetic:
  Accuracy:  0.95
  Precision: 0.96
  Recall:    0.97
  F1-Score:  0.97
  FPR:       0.12
  FNR:       0.03

Metrics for denial_service:
  Accuracy:  0.96
  Precision: 0.90
  Recall:    0.89
  F1-Score:  0.90
  FPR:       0.02
  FNR:       0.11

Metrics for front_running:
  Accuracy:  0.96
  Precision: 0.89
  Recall:    0.79
  F1-Score:  0.84
  FPR:       0.02
  FNR:       0.21

Metrics for reentrancy:
  Accuracy:  0.91
  Precision: 0.88
  Recall:    0.82
  F1-Score:  0.85
  FPR:       0.05
  FNR:       0.18

Metrics for time_manipulation:
  Accuracy:  0.96
  Precision: 0.88
  Recall:    0.48
  F1-Score:  0.62
  FPR:       0.00
  FNR:       0.52

Metrics for unchecked_low_calls:
  Accuracy:  0.93
  Pre